
test the implemented objective functions and constriants

In [1]:
from robot import Robot, MotomanRobot
from planning_scene import PlanningScene
from geometric_trajopt_ipopt import PoseTrajOpt
import numpy as np
import scipy as sp

In [2]:
# test the robot with the scene
# add environment collisions
pcd_total = []
# shelf-bottom
num_points = 1000
position = np.array([0.85, 0, 0.5])
half_size = np.array([0.175, 0.5, 0.5])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-top
num_points = 1000
position = np.array([0.85, 0, 1.42])
half_size = np.array([0.175, 0.5, 0.025])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-left
num_points = 1000
position = np.array([0.85, -0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-right
num_points = 1000
position = np.array([0.85, 0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-back
num_points = 1000
position = np.array([1.0, 0, 1.2])
half_size = np.array([0.025, 0.5, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
pcd_total = np.concatenate(pcd_total, axis=0)


torso_b1 = ["torso_joint_b1"]
left = [
    "arm_left_joint_1_s",
    "arm_left_joint_2_l",
    "arm_left_joint_3_e",
    "arm_left_joint_4_u",
    "arm_left_joint_5_r",
    "arm_left_joint_6_b",
    "arm_left_joint_7_t",
]
right = [
    "arm_right_joint_1_s",
    "arm_right_joint_2_l",
    "arm_right_joint_3_e",
    "arm_right_joint_4_u",
    "arm_right_joint_5_r",
    "arm_right_joint_6_b",
    "arm_right_joint_7_t",
]
robot_joint_names = right#torso_b1 + left + right
robot = MotomanRobot(selected_joint_names=robot_joint_names)
# scene = PlanningScene(robot, scene_pcd=pcd_total)
scene = PlanningScene(robot, scene_pcd=None)
scene.update_scene_pcd(pcd_total)


In [3]:
print(robot.selected_joint_limits)

[[-3.14159  3.14159]
 [-1.91986  1.91986]
 [-2.96706  2.96706]
 [-2.35619  2.35619]
 [-3.14159  3.14159]
 [-1.91986  1.91986]
 [-3.14159  3.14159]]


In [4]:
import transformations as tf
pos = np.array([0.9096643361780983, -0.22, 1.2246445275392399])
quat = np.array([0.99997882565292, 0.0020695812292800746, -0.005795312726922581, 0.0021164663332217232])
pose = tf.quaternion_matrix(quat)
pose[:3,3] = pos
n_waypoints = 10
start_q = np.zeros((len(robot.selected_joint_dofids)))
solver = PoseTrajOpt(robot, scene, start_q=start_q, target_link="arm_right_link_7_t", target_pose=pose, n_waypoints=n_waypoints)


lb shape:  (70,)
lb: 
[-3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159
 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986
 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706
 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619
 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159
 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986
 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159
 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159
 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159]
ub: 
[3.14159 1.91986 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159 1.91986
 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159 1.91986 2.96706 2.35619
 3.14159 1.91986 3.14159 3.14159 1.91986 2.96706 2.35619 3.14159 1.91986
 3.14159 3.14159 1.91986 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159
 1.91986 2.96706 2.35619 3.14159 1.91986 

In [5]:
qs = np.random.uniform(robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1], size=(2, len(robot_joint_names)))
# qs = np.linspace(start=qs[0,:], stop=qs[1,:], num=n_waypoints+1)
# solver.set_start_q(qs[0,:])
# qs = qs[1:,:]
# print(solver.start_q)
# print(qs)

step = (qs[1]-qs[0]) / (n_waypoints+1)
q0 = qs[0,:]
qs = []
solver.set_start_q(q0)
qs.append(q0+step)
for i in range(n_waypoints-1):
    qs.append(qs[-1]+step)
qs = np.array(qs)

### verify the objective function and gradient

In [6]:
# verify the objective function
print(solver.waypoint_distance_objective(qs))

diff = np.diff(qs, n=1, axis=0)
diff_length = np.linalg.norm(diff, ord=2, axis=1)
print(diff_length.sum()+np.linalg.norm(qs[0,:]-solver.start_q, ord=2))

3.5891323042834946
3.5891323042834946


In [7]:
length = np.linalg.norm(diff, ord=2, axis=1)
diff_normalized = diff / np.linalg.norm(diff, ord=2, axis=1)[:,np.newaxis]
print('diff: ')
for i in range(len(diff)):
    print('diff: ')
    print(diff[i])
    print('diff/length: ', diff[i]/length[i])
    print('diff_normalized: ')
    print(diff_normalized[i])
    print('length: ')
    print(np.linalg.norm(diff_normalized[i]))
gradient = solver.waypoint_distance_gradient(qs)
gradient.reshape((n_waypoints, -1))
# verfy that the gradient is correct by numerical differentiation

diff: 
diff: 
[ 0.05205369 -0.12761482 -0.08748253 -0.03049106 -0.11814135  0.28927197
  0.0600419 ]
diff/length:  [ 0.14503139 -0.35555899 -0.24374284 -0.08495383 -0.32916409  0.8059663
  0.16728806]
diff_normalized: 
[ 0.14503139 -0.35555899 -0.24374284 -0.08495383 -0.32916409  0.8059663
  0.16728806]
length: 
0.9999999999999999
diff: 
[ 0.05205369 -0.12761482 -0.08748253 -0.03049106 -0.11814135  0.28927197
  0.0600419 ]
diff/length:  [ 0.14503139 -0.35555899 -0.24374284 -0.08495383 -0.32916409  0.8059663
  0.16728806]
diff_normalized: 
[ 0.14503139 -0.35555899 -0.24374284 -0.08495383 -0.32916409  0.8059663
  0.16728806]
length: 
1.0
diff: 
[ 0.05205369 -0.12761482 -0.08748253 -0.03049106 -0.11814135  0.28927197
  0.0600419 ]
diff/length:  [ 0.14503139 -0.35555899 -0.24374284 -0.08495383 -0.32916409  0.8059663
  0.16728806]
diff_normalized: 
[ 0.14503139 -0.35555899 -0.24374284 -0.08495383 -0.32916409  0.8059663
  0.16728806]
length: 
1.0
diff: 
[ 0.05205369 -0.12761482 -0.08748253 -

array([[-2.77555756e-17,  5.55111512e-17,  5.55111512e-17,
         1.38777878e-17,  5.55111512e-17, -1.11022302e-16,
         1.11022302e-16],
       [-2.77555756e-17,  5.55111512e-17, -1.38777878e-16,
         1.38777878e-17,  5.55111512e-17, -1.11022302e-16,
        -2.77555756e-17],
       [ 0.00000000e+00, -1.66533454e-16,  3.33066907e-16,
        -3.05311332e-16,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.0000000

In [8]:
def naive_finite_diff(func, x, eps=1e-8):
    # given a function and the point for differentiation, compute the naive finite diff
    # x is a numpy array of shape (n,)
    diff = eps
    grad = np.zeros(x.shape)
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += diff
        x_minus = x.copy()
        x_minus[i] -= diff
        grad[i] = (func(x_plus) - func(x_minus)) / (2*diff)
    return grad

In [9]:
print(solver.start_q)
print(qs)
diff = qs[1,:] - qs[0,:]
diff = diff / np.linalg.norm(diff, ord=2)


[ 1.43657783  0.98839971 -0.29268688  1.11007959  2.63635142 -1.60887867
 -0.57934103]
[[ 1.48863152  0.86078488 -0.38016941  1.07958853  2.51821007 -1.3196067
  -0.51929913]
 [ 1.5406852   0.73317006 -0.46765195  1.04909748  2.40006872 -1.03033473
  -0.45925723]
 [ 1.59273889  0.60555523 -0.55513448  1.01860642  2.28192738 -0.74106276
  -0.39921534]
 [ 1.64479257  0.47794041 -0.64261701  0.98811537  2.16378603 -0.45179079
  -0.33917344]
 [ 1.69684626  0.35032558 -0.73009954  0.95762431  2.04564469 -0.16251882
  -0.27913154]
 [ 1.74889994  0.22271076 -0.81758207  0.92713326  1.92750334  0.12675315
  -0.21908964]
 [ 1.80095363  0.09509593 -0.9050646   0.8966422   1.809362    0.41602512
  -0.15904774]
 [ 1.85300732 -0.03251889 -0.99254714  0.86615115  1.69122065  0.70529709
  -0.09900584]
 [ 1.905061   -0.16013372 -1.08002967  0.83566009  1.57307931  0.99456906
  -0.03896395]
 [ 1.95711469 -0.28774854 -1.1675122   0.80516904  1.45493796  1.28384103
   0.02107795]]


In [10]:
numerical_gradient = naive_finite_diff(solver.waypoint_distance_objective, qs.flatten())
gradient = solver.waypoint_distance_gradient(qs)
print('numerical gradient: ', numerical_gradient.reshape((n_waypoints, -1)))
print('gradient: ', gradient.reshape((n_waypoints, -1)))
print('difference: ', np.linalg.norm(numerical_gradient-gradient))

numerical gradient:  [[-2.22044605e-08  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  4.44089210e-08  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  2.22044605e-08  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-2.22044605e-08  2.22044605e-08  0.000000

In [11]:
# randomly sample nodes to test
qs_new = np.random.uniform(robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1], size=(n_waypoints+1, len(robot_joint_names)))
solver.set_start_q(qs_new[0,:])
qs_new = qs_new[1:,:]
numerical_gradient = naive_finite_diff(solver.waypoint_distance_objective, qs_new.flatten())
gradient = solver.waypoint_distance_gradient(qs_new)
print('numerical gradient: ', numerical_gradient.reshape((n_waypoints, -1)))
print('gradient: ', gradient.reshape((n_waypoints, -1)))
print('difference: ', np.linalg.norm(numerical_gradient-gradient))
solver.set_start_q(q0)

numerical gradient:  [[-0.41585402  0.13418209  0.78269871  0.78542968 -0.09272441  0.4847962
  -1.17139045]
 [ 0.2041066   0.44134012 -1.34838665 -0.81358955 -0.62940764 -0.56242406
   0.54512128]
 [-0.20736231 -0.14409345  0.6011998   0.62779684  0.73651556  1.16461116
   0.32697081]
 [ 0.17489192 -0.31266119  0.67257169 -0.58318292 -0.95755404 -0.95439994
  -0.76843847]
 [ 0.64186203  0.00286526 -0.57351919  0.52489959  1.04611182  0.03895053
   0.90044487]
 [-1.10013758  0.05374972 -0.80343412 -0.04703971 -0.76310904  0.00672244
  -0.72382491]
 [ 0.07444072  0.23109905  1.14881011 -0.53052531  0.51572613  0.59847594
  -0.35809862]
 [ 1.02049285 -0.18483064 -0.06667769  0.74018764 -0.11016077 -0.68994801
   1.10469927]
 [-1.60136295  0.19454873 -0.19912996 -0.31703671 -0.04121219  0.44005866
  -0.46621871]
 [ 0.95778852 -0.13988846 -0.09673151 -0.03303064 -0.06633591 -0.20219311
  -0.08568399]]
gradient:  [[-0.41585391  0.13418176  0.78269881  0.78542962 -0.09272427  0.48479615
  -1

In [12]:
solver.terminal_pose_objective(qs)
solver.terminal_pose_gradient(qs).reshape((n_waypoints, -1))

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [-0.49697379, -0.7646

In [13]:

def naive_finite_diff_jacobian(func, x, eps=1e-8):
    # given a function and the point for differentiation, compute the naive finite diff
    # func returns array of shape (M,)  #(M can be a list)
    # x is a numpy array of shape (n,)
    # result of shape (M,n)
    func_shape = list(func(x).shape)
    x_shape = list(x.shape)
    jacobian = np.zeros(list(func(x).shape)+list(x.shape)).reshape((np.prod(func_shape), np.prod(x_shape)))  # first flatten the func shape
    for i in range(np.prod(x_shape)):
        x_plus = x.copy()
        x_plus.flat[i] += eps
        x_minus = x.copy()
        x_minus.flat[i] -= eps
        
        deriv = (func(x_plus) - func(x_minus)) / (2*eps)
        jacobian[:,i] = deriv.flatten()
    jacobian = jacobian.reshape(func_shape+x_shape)
    return jacobian

In [14]:
# check robot.get_link_analytical_jacobian(link_name)
link_name = "arm_right_link_7_t"
def link_pose(q, link_name):
    robot.set_selected_joint_values(q)
    return robot.get_link_pose(link_name)

q = qs[-1].copy()
# numerical jacobian of the pose
numerical_jacobian = naive_finite_diff_jacobian(lambda q: link_pose(q, link_name), q)
# analytical jacobian of the pose
analytical_jacobian = robot.get_link_analytical_jacobian(link_name)
pose = robot.get_link_pose(link_name)
# check the difference
for i in range(len(robot.selected_joint_dofids)):
    print('numerical jacobian: ')
    print(numerical_jacobian[:,:,i])
    print('analytical jacobian: ')
    print(analytical_jacobian[:,:,i])
    print('difference: ')
    print(np.linalg.norm(numerical_jacobian[:,:,i]-analytical_jacobian[:,:,i]))


numerical jacobian: 
[[-8.41491432e-01 -3.49924351e-01 -4.11637124e-01  2.76829026e-01]
 [-2.22044605e-08 -2.22044605e-08  5.55111512e-09  5.55111512e-09]
 [-2.15945439e-01 -4.80557666e-01  8.49959941e-01  2.50901644e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]
analytical jacobian: 
[[-8.41491439e-01 -3.49924344e-01 -4.11637111e-01  2.76829034e-01]
 [ 4.79495151e-17  1.06705240e-16 -1.88729018e-16 -5.57113562e-17]
 [-2.15945418e-01 -4.80557681e-01  8.49959935e-01  2.50901643e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]
difference: 
4.5573065019765916e-08
numerical jacobian: 
[[ 0.18659573 -0.30297881 -0.12389326 -0.22850961]
 [ 0.69811156  0.14307093  0.70154894 -0.16189254]
 [ 0.4587393  -0.74486314 -0.30458735 -0.56178316]
 [ 0.          0.          0.          0.        ]]
analytical jacobian: 
[[ 0.18659574 -0.30297881 -0.12389325 -0.22850963]
 [ 0.69811157  0.14307095  0.70154896 -0.16189254]
 [ 0.45873932 -0.74486315 -0.3045

In [15]:
# check as eps goes down if they are similar
# check robot.get_link_analytical_jacobian(link_name)
link_name = "arm_right_link_7_t"
def link_pose(q, link_name):
    robot.set_selected_joint_values(q)
    return robot.get_link_pose(link_name)

q = qs[-1].copy()
for eps in [1e-10, 1e-12, 1e-14, 1e-16]:
    print('eps: ', eps)
    # numerical jacobian of the pose
    numerical_jacobian = naive_finite_diff_jacobian(lambda q: link_pose(q, link_name), q, eps=eps)
    # analytical jacobian of the pose
    analytical_jacobian = robot.get_link_analytical_jacobian(link_name)
    # check the difference
    print('diff: ')
    print(numerical_jacobian - analytical_jacobian)
    print('norm: ', np.linalg.norm(numerical_jacobian - analytical_jacobian))
    # check position difference
    print('position diff:')
    print(numerical_jacobian[:3,3,:] - analytical_jacobian[:3,3,:])
    print('norm:', np.linalg.norm(numerical_jacobian[:3,3,:] - analytical_jacobian[:3,3,:]))
    print('rotation diff:')
    print(numerical_jacobian[:3,:3,:] - analytical_jacobian[:3,:3,:])
    print('norm:', np.linalg.norm(numerical_jacobian[:3,:3,:] - analytical_jacobian[:3,:3,:]))


eps:  1e-10
diff: 
[[[-1.54389215e-06  5.52963955e-07  8.90051516e-07 -1.31550406e-06
    3.63535035e-07  2.32348891e-07 -4.18292017e-07]
  [ 7.96822751e-08  2.55734641e-06  1.40289465e-06 -9.92159248e-07
   -6.91921401e-07  6.39914979e-07 -7.26651724e-07]
  [ 2.75464783e-07  2.35237206e-06  1.05358864e-06 -1.85380183e-06
   -4.49122503e-07 -2.97650796e-07 -4.16333634e-16]
  [ 9.13650347e-07  3.30927047e-07 -3.85649949e-07  3.27444114e-07
    1.91288423e-07  2.86931034e-07  0.00000000e+00]]

 [[-2.77555756e-07  1.38494272e-06  1.24568198e-06  1.71494185e-06
    1.07615920e-07  1.12079654e-06 -1.04795269e-06]
  [ 1.11022302e-06  2.93844084e-06  1.32135426e-06 -2.21770579e-06
    2.27473737e-06 -2.15177557e-07 -1.43996627e-07]
  [-5.55111512e-07 -1.52486334e-06 -5.15522281e-07  1.84138693e-06
    1.69071308e-07 -2.62502979e-07 -2.77555755e-07]
  [ 5.55111512e-07  3.73921235e-07 -1.64200761e-07  3.97951372e-07
    1.16411673e-07 -1.29505804e-07  0.00000000e+00]]

 [[-1.71540211e-07  1.511

In [16]:
# tested: the position parts are correct (by turning off the rotation part in the solver computation)
# tested: in test_so3_se3.py, verified that the derivative of log_v(R) is correct
numerical_gradient = naive_finite_diff(solver.terminal_pose_objective, qs.flatten(), eps=1e-8)
gradient = solver.terminal_pose_gradient(qs)
print('numerical gradient: ', numerical_gradient)
print('gradient: ', gradient)
print('diff: ', numerical_gradient - gradient)
print('norm:', np.linalg.norm(numerical_gradient - gradient))

numerical gradient:  [ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.         -0.49697375 -0.76462556 -0.43711195
  0.30968774  0.88343572 -0.4194002   0.54750111]
gradient:  [ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.


In [17]:
# separate different steps to verify the correctness of the gradient
# 1. check d ||log_v(R)|| / dR
# 2. check d d(R_target*R^T) / dR
import so3

# step 1
def terminal_pose_gradient(solver, qs):
    """
    return the gradient of the terminal pose objective.
    d J_p / d q = d ||x_p(q_N) - target_position|| / d q = 
                    (x_p(q_N) - target_position)/||x_p(q_N) - target_position||*J_p(q_N)
    d J_r / d q = d ||log(R_target * R_N^T)_V|| / d q = 
                    (log(R_target * R_N^T)_V)/||log(R_target * R_N^T)_V||*
                    d log(R_target * R_N^T)_V / d (R_target*R_N^T) * d (R_target*R_N^T) / d R_N * d R_N / d q
    """
    qs = np.array(qs).reshape(solver.n_waypoints, solver.ndof)
    robot =solver.robot
    target_link = solver.target_link
    target_pose = solver.target_pose
    robot.set_selected_joint_values(qs[-1])
    pose = robot.get_link_pose(target_link)
    # * get the gradient of the orientation difference. Shape: nv
    # get orientation difference
    link_jac = robot.get_link_analytical_jacobian(target_link)  # 4x4xnv
    R = pose[:3, :3]
    rot_diff = target_pose[:3,:3].dot(R.T)
    print('rot_diff: ', rot_diff)
    w_diff = so3.log_quaternion(rot_diff)  # shape: 3
    jac1 = so3.Dx_log_x_quaternion(rot_diff).reshape((3,3,3)) # shape: 3x9
    # d ||log(R_target * R_N^T)_V|| / d (R_target*R_N^T) = 
    # d ||log(R_target*R_N^T)_V|| / d log(R_target*R_N^T)_V * d log(R_target*R_N^T)_V / d (R_target*R_N^T)
    d_w_norm_dR = w_diff / np.linalg.norm(w_diff) @ jac1  # shape: 3x3
    # d (R_target * R_N^T) / d R_N
    # TODO: verify the following
    dR_dR = np.zeros((3,3,3,3))
    dR_dR[:,0,0,:] = target_pose[:3,:3]
    dR_dR[:,1,1,:] = target_pose[:3,:3]
    dR_dR[:,2,2,:] = target_pose[:3,:3]
    jac_r = np.tensordot(d_w_norm_dR, dR_dR, axes=2) # shape: 3x3
    rotation_diff_gradient = np.tensordot(jac_r, link_jac[:3,:3,:], axes=2) # shape: nv
    # # TODO: sparse gradient
    gradient = np.zeros((solver.n_waypoints*solver.ndof))
    gradient[-solver.ndof:] = rotation_diff_gradient
    dw_dq = np.tensordot(jac1, dR_dR, axes=2)  # shape: 3x3x3
    dw_dq = np.tensordot(dw_dq, link_jac[:3,:3,:], axes=2)  # shape: 3xnv
    return d_w_norm_dR, dR_dR, dw_dq, gradient

def get_log_v_norm(R):
    R = np.array(R).reshape((3,3))
    return np.linalg.norm(so3.log_quaternion(R))

def get_v_norm(v):
    return np.linalg.norm(v)

In [18]:
def naive_finite_diff(func, x, eps=1e-8):
    # given a function and the point for differentiation, compute the naive finite diff
    # x is a numpy array of shape (n,)
    diff = eps
    grad = np.zeros(x.shape)
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += diff
        x_minus = x.copy()
        x_minus[i] -= diff
        grad[i] = (func(x_plus) - func(x_minus)) / (2*diff)
    return grad

In [19]:
# check the gradient: d ||v|| / dv = v / ||v||
R_current = robot.get_link_pose(solver.target_link)[:3,:3]
R = solver.target_pose[:3,:3] @ R_current.T
v = so3.log_quaternion(R)
numerical_gradient = naive_finite_diff(get_v_norm, v, eps=1e-8)
gradient = v / np.linalg.norm(v)
print('numerical gradient: ', numerical_gradient)
print('gradient: ', gradient)
jac1 = so3.Dx_log_x_quaternion(R).reshape((3,3,3))  # d log(R)/dR
# check d log(R)/dR
numerical_jacobian = naive_finite_diff_jacobian(so3.log_quaternion, R, eps=1e-8)
# print('numerical_jacobian: ', numerical_jacobian)
# print('jac1: ', jac1)
print('diff: ', np.linalg.norm(numerical_jacobian-jac1))

d_w_norm_dR = gradient @ jac1
print('d_w_norm_dR: ', d_w_norm_dR)
# check the gradient: d ||log_v(R)|| / dR
numerical_gradient_dR = naive_finite_diff(get_log_v_norm, R.flatten(), eps=1e-5).reshape((3,3))
print('numerical_gradient_dR: ', numerical_gradient_dR)

numerical gradient:  [ 0.78220079  0.31561358 -0.53716853]
gradient:  [ 0.78220078  0.31561356 -0.53716852]
diff:  7.086457402640511e-08
d_w_norm_dR:  [[ 0.00405709 -0.0467032   0.69442028]
 [ 0.00163701  0.88976482  0.06367414]
 [-0.00278617 -0.18444754  0.67302038]]
numerical_gradient_dR:  [[ 0.00405709  0.00163701 -0.00278617]
 [ 0.00163701 -0.00405709  0.63922206]
 [-0.00278617 -0.63922206 -0.00405709]]


In [20]:
# check step 1 by taylor expansion
# f(R') = f(R) + df/dR(R) * (R'-R)
# collect data, so that we have
# (f(R') - f(R))_i = df/dR(R) * (R'-R)_i
# then we compute df/dR(R) using least square
f_diffs = []
As = []
d_w_norm_dR, dR_dR, dw_dq, gradient = terminal_pose_gradient(solver, qs)
for i in range(20):
    angle = 1e-8 * np.pi / 180
    axis = np.random.random((3))
    axis = axis / np.linalg.norm(axis)
    d_R = tf.rotation_matrix(angle, axis)[:3,:3]
    R_current = robot.get_link_pose(solver.target_link)[:3,:3]
    R = solver.target_pose[:3,:3] @ R_current.T
    R_prime = d_R.dot(R)
    f_diff = get_log_v_norm(R_prime) - get_log_v_norm(R)
    f_diff_approx = np.tensordot(d_w_norm_dR, R_prime-R, axes=2)
    f_diffs.append(f_diff)
    As.append((R_prime-R).flatten())
f_diffs = np.array(f_diffs)  # shape: n
As = np.array(As)  # shape: nx9
J = d_w_norm_dR.flatten() # shape: 9
# least square: As * J = f_diffs
# J = As^{-1}*f_diffs
J_approx = np.linalg.pinv(As) @ f_diffs
d_w_norm_dR_numerical = J.reshape((3,3))
print('d_w_norm_dR: ', d_w_norm_dR)
print('d_w_norm_dR_numerical: ', d_w_norm_dR_numerical)

# this verifies that d ||log_v(R)|| / dR is correct

rot_diff:  [[ 0.22372724  0.50243064 -0.83517039]
 [ 0.48499704 -0.80065761 -0.35174602]
 [-0.8454135  -0.32636    -0.42280629]]
d_w_norm_dR:  [[ 0.00405709 -0.0467032   0.69442028]
 [ 0.00163701  0.88976482  0.06367414]
 [-0.00278617 -0.18444754  0.67302038]]
d_w_norm_dR_numerical:  [[ 0.00405709 -0.0467032   0.69442028]
 [ 0.00163701  0.88976482  0.06367414]
 [-0.00278617 -0.18444754  0.67302038]]


In [21]:
# now verify d (R_target * R_N^T) / d R_N
R_target = solver.target_pose[:3,:3]
R_N = robot.get_link_pose(solver.target_link)[:3,:3]
R_diff = R_target @ R_N.T
print('R_diff: ', R_diff)  # check if it is the same as in the function
dR_dR_numerical = naive_finite_diff_jacobian(lambda R: R_target @ R.T, R_N, eps=1e-8)
print('dR_dR: ', dR_dR)
print('dR_dR_numerical: ', dR_dR_numerical)
print('diff: ', np.linalg.norm(dR_dR-dR_dR_numerical)) # verified that it is correct

R_diff:  [[ 0.22372724  0.50243064 -0.83517039]
 [ 0.48499704 -0.80065761 -0.35174602]
 [-0.8454135  -0.32636    -0.42280629]]
dR_dR:  [[[[ 0.99992387 -0.00425683 -0.01158162]
   [ 0.          0.          0.        ]
   [ 0.          0.          0.        ]]

  [[ 0.          0.          0.        ]
   [ 0.99992387 -0.00425683 -0.01158162]
   [ 0.          0.          0.        ]]

  [[ 0.          0.          0.        ]
   [ 0.          0.          0.        ]
   [ 0.99992387 -0.00425683 -0.01158162]]]


 [[[ 0.00420886  0.99998247 -0.00416361]
   [ 0.          0.          0.        ]
   [ 0.          0.          0.        ]]

  [[ 0.          0.          0.        ]
   [ 0.00420886  0.99998247 -0.00416361]
   [ 0.          0.          0.        ]]

  [[ 0.          0.          0.        ]
   [ 0.          0.          0.        ]
   [ 0.00420886  0.99998247 -0.00416361]]]


 [[[ 0.01159914  0.00411454  0.99992426]
   [ 0.          0.          0.        ]
   [ 0.          0.          

In [22]:
# verify d R_N / d q
def link_pose(q, link_name):
    robot.set_selected_joint_values(q)
    return robot.get_link_pose(link_name)
robot.set_selected_joint_values(qs[-1])
pose = robot.get_link_pose(solver.target_link)
# * get the gradient of the orientation difference. Shape: nv
# get orientation difference
link_jac = robot.get_link_analytical_jacobian(solver.target_link)  # 4x4xnv
# * get the numeircal jacobian of the pose
numerical_link_jacobian = naive_finite_diff_jacobian(lambda q: link_pose(q, solver.target_link), qs[-1], eps=1e-8)
robot.set_selected_joint_values(qs[-1]) # reset

print('analytical jacobian:')
print(link_jac)
print('numerical jacobian:')
print(numerical_link_jacobian)
print('diff: ', np.linalg.norm(link_jac-numerical_link_jacobian))
# this verifies that the dR_N / dq is correct

analytical jacobian:
[[[-8.41491443e-01  1.86595741e-01 -6.76706249e-01  5.51014439e-01
    1.53195426e-01  8.49771132e-01  4.80557679e-01]
  [-3.49924336e-01 -3.02978813e-01 -5.46925020e-01 -4.46698907e-01
    7.53904298e-01 -1.79140885e-02 -2.15945423e-01]
  [-4.11637111e-01 -1.23893250e-01 -4.81152839e-01 -1.12564769e-01
    4.65170574e-01  2.05769033e-01  4.16333634e-16]
  [ 2.76829034e-01 -2.28509626e-01  1.06016554e-01 -2.22379526e-01
    7.21014389e-02  3.18942001e-02  0.00000000e+00]]

 [[ 4.79495162e-17  6.98111571e-01 -1.46746639e-01 -7.30019428e-01
   -2.20947257e-01  3.28747574e-01 -8.04125162e-01]
  [ 1.06705240e-16  1.43070948e-01 -1.63745340e-01 -5.65026362e-01
    1.75136518e-01 -6.93035209e-03 -4.95237041e-01]
  [-1.88729018e-16  7.01548956e-01  1.79421161e-01  2.82279302e-01
   -7.61062216e-01  5.12075089e-01 -1.11022302e-15]
  [-5.57113562e-17 -1.61892540e-01  9.55576150e-02  3.20408302e-01
   -1.17964643e-01  7.93716389e-02  0.00000000e+00]]

 [[-2.15945423e-01  4.5

In [23]:
def terminal_pose_objective(solver, qs):
    """
    return the distance between the last waypoint and the target pose.
    """
    qs = np.array(qs).reshape(solver.n_waypoints, solver.ndof)
    robot = solver.robot
    target_link = solver.target_link
    target_pose = solver.target_pose
    robot.set_selected_joint_values(qs[-1])
    pose = robot.get_link_pose(target_link)
    # get position difference
    position_diff = pose[:3,3] - target_pose[:3,3]
    # get orientation difference
    R = pose[:3, :3]
    rot_diff = so3.log_quaternion(np.dot(target_pose[:3, :3], R.T))
    # TODO: add weight for the position and rotation diff
    # return np.linalg.norm(position_diff) + np.linalg.norm(rot_diff)
    return np.linalg.norm(rot_diff)

def terminal_pose_objective_selected(solver, q):
    """
    return the distance between the last waypoint and the target pose.
    """
    robot = solver.robot
    target_link = solver.target_link
    target_pose = solver.target_pose
    robot.set_selected_joint_values(q)
    pose = robot.get_link_pose(target_link)
    # get position difference
    position_diff = pose[:3,3] - target_pose[:3,3]
    # get orientation difference
    R = pose[:3, :3]
    rot_diff = so3.log_quaternion(np.dot(target_pose[:3, :3], R.T))
    # TODO: add weight for the position and rotation diff
    # return np.linalg.norm(position_diff) + np.linalg.norm(rot_diff)
    return np.linalg.norm(rot_diff)


In [24]:
def terminal_rot(solver, q):
    robot = solver.robot
    target_link = solver.target_link
    target_pose = solver.target_pose
    robot.set_selected_joint_values(q)
    pose = robot.get_link_pose(target_link)
    # get position difference
    position_diff = pose[:3,3] - target_pose[:3,3]
    # get orientation difference
    R = pose[:3, :3]
    rot_diff = so3.log_quaternion(np.dot(target_pose[:3, :3], R.T))
    # TODO: add weight for the position and rotation diff
    # return np.linalg.norm(position_diff) + np.linalg.norm(rot_diff)
    return rot_diff



In [25]:
# * check why the gradient is different. Above verifies that each chain jacobian is correct
numerical_gradient = naive_finite_diff(lambda q: terminal_pose_objective(solver, q), qs.flatten(), eps=1e-5)
# gradient = terminal_pose_gradient(solver, qs)
print('numerical gradient: ', numerical_gradient)
print('gradient: ', gradient)
print('diff: ', numerical_gradient - gradient)
print('norm:', np.linalg.norm(numerical_gradient - gradient))

numerical gradient:  [ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.         -0.31561356 -0.92694926 -0.36020833
  0.26724007  0.89398524 -0.32638064  0.54750108]
gradient:  [ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.


In [26]:
def terminal_pose_gradient_selected(solver, q):
    robot =solver.robot
    target_link = solver.target_link
    target_pose = solver.target_pose
    robot.set_selected_joint_values(q)
    pose = robot.get_link_pose(target_link)
    # * get the gradient of the orientation difference. Shape: nv
    # get orientation difference
    link_jac = robot.get_link_analytical_jacobian(target_link)  # 4x4xnv
    R = pose[:3, :3]
    rot_diff = target_pose[:3,:3].dot(R.T)
    w_diff = so3.log_quaternion(rot_diff)  # shape: 3
    jac1 = so3.Dx_log_x_quaternion(rot_diff).reshape((3,3,3)) # shape: 3x9
    # d ||log(R_target * R_N^T)_V|| / d (R_target*R_N^T) = 
    # d ||log(R_target*R_N^T)_V|| / d log(R_target*R_N^T)_V * d log(R_target*R_N^T)_V / d (R_target*R_N^T)
    d_w_norm_dR = (w_diff / np.linalg.norm(w_diff)) @ jac1  # shape: 3x3
    # d (R_target * R_N^T) / d R_N
    # TODO: verify the following
    dR_dR = np.zeros((3,3,3,3))
    dR_dR[:,0,0,:] = target_pose[:3,:3]
    dR_dR[:,1,1,:] = target_pose[:3,:3]
    dR_dR[:,2,2,:] = target_pose[:3,:3]
    jac_r = np.tensordot(d_w_norm_dR, dR_dR, axes=2) # shape: 3x3
    rotation_diff_gradient = np.tensordot(jac_r, link_jac[:3,:3,:], axes=2) # shape: nv
    # # TODO: sparse gradient
    gradient = rotation_diff_gradient
    # gradient = np.zeros((solver.ndof))
    # gradient[-solver.ndof:] = rotation_diff_gradient
    dw_dq = np.tensordot(jac1, dR_dR, axes=2)  # shape: 3x3x3
    dw_dq = np.tensordot(dw_dq, link_jac[:3,:3,:], axes=2)  # shape: 3xnv

    # compare two methods to compute dw_norm_dq
    # grad1 = (w_diff / np.linalg.norm(w_diff)) @ (so3.Dx_log_x_quaternion(rot_diff).reshape((3,3,3)))
    # grad1 = (w_diff / np.linalg.norm(w_diff)) @ (so3.Dx_log_x_quaternion(rot_diff))
    
    # print('grad1.shape: ', grad1.shape)
    # grad1 = grad1.reshape((3,3))
    grad1 = np.tensordot(w_diff / np.linalg.norm(w_diff), so3.Dx_log_x_quaternion(rot_diff).reshape((3,3,3)), axes=1)
    grad1 = np.tensordot(grad1, dR_dR, axes=2)
    grad1 = np.tensordot(grad1, link_jac[:3,:3,:], axes=2)
    print('grad1: ')
    print(grad1)
    grad2 = so3.Dx_log_x_quaternion(rot_diff).reshape((3,3,3))
    grad2 = np.tensordot(grad2, dR_dR, axes=2)
    grad2 = np.tensordot(grad2, link_jac[:3,:3,:], axes=2)
    grad2 = (w_diff / np.linalg.norm(w_diff)) @ grad2
    print('grad2: ')
    print(grad2)
    return d_w_norm_dR, dR_dR, dw_dq, gradient


In [27]:
def R_to_w_norm(R):
    rot_diff = so3.log_quaternion(R.reshape((3,3)))
    return np.linalg.norm(rot_diff)

dw_norm_dR_selected, dR_dR_selected, dw_dq_selected, d_norm_dq_terminal = terminal_pose_gradient_selected(solver, qs[-1])
# compare dw_norm_dR
R_current = robot.get_link_pose(solver.target_link)[:3,:3]
R = solver.target_pose[:3,:3] @ R_current.T
w = so3.log_quaternion(R)
dw_norm_dw = w / np.linalg.norm(w)
dw_norm_dq = dw_norm_dw.dot(dw_dq_selected)
dw_norm_dq_numerical = naive_finite_diff(lambda q: terminal_pose_objective_selected(solver, q), qs[-1])


print('diff: ', np.linalg.norm(dw_norm_dq-dw_norm_dq_numerical))

grad1: 
[-0.31561356 -0.92694926 -0.36020833  0.26724007  0.89398524 -0.32638064
  0.54750108]
grad2: 
[-0.31561356 -0.92694926 -0.36020833  0.26724007  0.89398524 -0.32638064
  0.54750108]
diff:  6.58064354931859e-08


diff:  6.266048040348744e-08
